# A model that remembers you

This notebook answers a common question: **does the model know me and remember our last chats?**

Short answer: the model itself does **not** remember you between separate requests. It only sees the `messages` you send each time.

Then we will build a simple memory system in the browser. The memory will live in a small SQLite database saved in this browser only.

## Step 1: Get set up

This loads WebLLM, checks for WebGPU, and defines helpers for loading models.

In [ ]:
import js
import json
import math
import sqlite3
from datetime import datetime
from pyodide.ffi import to_js, create_proxy
from IPython.display import display

if getattr(js.navigator, "gpu", None):
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

chat_engine = None
embed_engine = None
loaded_chat_model = None
loaded_embed_model = None


def js_obj(d):
    return to_js(d, dict_converter=js.Object.fromEntries)


def progress_bar():
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    return status, create_proxy(show)


async def load_chat_model(model_id):
    global chat_engine
    status, progress = progress_bar()

    async def attempt(chat_options=None):
        global chat_engine
        if chat_engine is None:
            if chat_options:
                chat_engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress}), js_obj(chat_options)
                )
            else:
                chat_engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress})
                )
        else:
            chat_engine.setInitProgressCallback(progress)
            if chat_options:
                await chat_engine.reload(model_id, js_obj(chat_options))
            else:
                await chat_engine.reload(model_id)

    try:
        await attempt()
    except Exception as error:
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await attempt({"sliding_window_size": -1})
        else:
            raise


async def load_embedding_model(model_id):
    global embed_engine
    status, progress = progress_bar()
    if embed_engine is None:
        embed_engine = await webllm.CreateMLCEngine(
            model_id, js_obj({"initProgressCallback": progress})
        )
    else:
        embed_engine.setInitProgressCallback(progress)
        await embed_engine.reload(model_id)


async def chat_completion(messages, **settings):
    if loaded_chat_model is None:
        return {"error": "No chat model is loaded yet. Run Step 3."}
    request = {
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 300,
        "frequency_penalty": 0.5,
    }
    request.update(settings)
    reply = await chat_engine.chat.completions.create(js_obj(request))
    return reply.to_py()


## Step 2: Choose your models

We use one chat model for conversation and, later, one embedding model for optional meaning-based search.

An **embedding** is a list of numbers that tries to capture the meaning of a piece of text. Similar texts should end up with similar embeddings.

In [ ]:
CHAT_MODEL_ID = "gemma3-1b-it-q4f16_1-MLC"
EMBED_MODEL_ID = "snowflake-arctic-embed-m-q0f32-MLC-b4"

print("Chat model:     " + CHAT_MODEL_ID)
print("Embedding model:" + EMBED_MODEL_ID)

## Step 3: Load the chat model

If the chat model is not already saved in your browser, this step will download it first.

In [ ]:
await load_chat_model(CHAT_MODEL_ID)
loaded_chat_model = CHAT_MODEL_ID
print(loaded_chat_model + " is loaded and ready.")

## Step 4: Show that the model does not remember across separate requests

In the first request, we tell the model a name. In the second request, we ask for the name again **without** sending the earlier message. If the model answers correctly, that is luck or guessing, not memory.

In [ ]:
first_request = [
    {"role": "user", "content": "My name is Priya. Please say hello and remember it for later."}
]

second_request = [
    {"role": "user", "content": "What is my name? If you do not know, say that clearly."}
]

first_reply = await chat_completion(first_request)
second_reply = await chat_completion(second_request)

if 'error' in first_reply:
    print(first_reply['error'])
else:
    print("First reply:")
    print(first_reply['choices'][0]['message']['content'])
    print()
    print("Second reply, in a fresh request:")
    print(second_reply['choices'][0]['message']['content'])

## Step 5: Show that the model can use what is inside `messages`

Now we send the earlier user message and the assistant reply back to the model. This is the ordinary way chat apps create the feeling of memory.

In [ ]:
with_history = [
    first_request[0],
    {"role": "assistant", "content": first_reply['choices'][0]['message']['content']},
    {"role": "user", "content": "What is my name?"},
]

history_reply = await chat_completion(with_history)
if 'error' in history_reply:
    print(history_reply['error'])
else:
    print(history_reply['choices'][0]['message']['content'])

## Step 6: Create a memory database in this browser

We will store each exchange in a small SQLite database file. This file lives in the JupyterLite file system for **this browser only**. If you use another browser or clear this browser's data, the memory will not come with you.

In [ ]:
DB_PATH = "chat_memory.sqlite3"
conn = sqlite3.connect(DB_PATH)
conn.execute("""
CREATE TABLE IF NOT EXISTS conversations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    created_at TEXT NOT NULL,
    label TEXT NOT NULL,
    user_text TEXT NOT NULL,
    assistant_text TEXT NOT NULL,
    embedding_json TEXT
)
""")
conn.commit()
print("Database ready at " + DB_PATH)

## Step 7: Save chats into the database

The next helper sends one user message to the model, saves both sides of the exchange, and returns the assistant's reply.

In [ ]:
def save_memory(label, user_text, assistant_text, embedding=None):
    conn.execute(
        "INSERT INTO conversations (created_at, label, user_text, assistant_text, embedding_json) VALUES (?, ?, ?, ?, ?)",
        (
            datetime.utcnow().isoformat(timespec="seconds"),
            label,
            user_text,
            assistant_text,
            json.dumps(embedding) if embedding is not None else None,
        ),
    )
    conn.commit()


async def chat_and_store(user_text, label="general"):
    reply = await chat_completion([{"role": "user", "content": user_text}])
    if 'error' in reply:
        return reply['error']
    assistant_text = reply['choices'][0]['message']['content']
    save_memory(label, user_text, assistant_text)
    return assistant_text

Try saving a few short chats. Use labels if you want to group them, like `school`, `hobbies`, or `travel`.

In [ ]:
MEMORY_LABEL = "school"
USER_TEXT = "I am studying biology and I like flash cards. Give me one study tip."

result = await chat_and_store(USER_TEXT, label=MEMORY_LABEL)
print(result)

## Step 8: Look at the saved memory

This prints the newest saved chats from the database.

In [ ]:
rows = conn.execute(
    "SELECT id, created_at, label, user_text, assistant_text FROM conversations ORDER BY id DESC LIMIT 5"
).fetchall()

if rows:
    for row in rows:
        print(f"#{row[0]} | {row[1]} | {row[2]}")
        print("  user:      " + row[3])
        print("  assistant: " + row[4])
        print()
else:
    print("No saved chats yet. Run Step 7 first.")

## Step 9: Use the last few chats as memory

A simple memory system is: take the last few saved exchanges and add them to the new `messages` list.

In [ ]:
def load_recent_memory(limit=3):
    rows = conn.execute(
        "SELECT user_text, assistant_text FROM conversations ORDER BY id DESC LIMIT ?",
        (limit,),
    ).fetchall()
    rows.reverse()
    messages = []
    for user_text, assistant_text in rows:
        messages.append({"role": "user", "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})
    return messages


NEW_QUESTION = "Based on what you know about me, suggest a better study plan."
messages = load_recent_memory(limit=3) + [{"role": "user", "content": NEW_QUESTION}]
reply = await chat_completion(messages)
if 'error' in reply:
    print(reply['error'])
else:
    print(reply['choices'][0]['message']['content'])

## Step 10: Optional meaning-based memory with embeddings

Looking up only the last few chats is simple, but sometimes an older chat is more relevant. We can search by **meaning** instead of only by time.

This step is optional because it needs a second model.

In [ ]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


async def embed_texts(texts):
    if loaded_embed_model is None:
        return {"error": "No embedding model is loaded yet. Run Step 11."}
    reply = await embed_engine.embeddings.create({"input": texts})
    return [item.embedding for item in reply.data]


def combined_text(user_text, assistant_text):
    return "[CLS] " + user_text + " \n " + assistant_text + " [SEP]"

## Step 11: Load the embedding model

This model turns text into vectors. It may take a little while to download the first time.

In [ ]:
await load_embedding_model(EMBED_MODEL_ID)
loaded_embed_model = EMBED_MODEL_ID
print(loaded_embed_model + " is loaded and ready.")

## Step 12: Add embeddings to saved chats

This fills in missing embedding vectors for any saved memories that do not have them yet.

In [ ]:
rows = conn.execute(
    "SELECT id, user_text, assistant_text FROM conversations WHERE embedding_json IS NULL ORDER BY id"
).fetchall()

if not rows:
    print("All saved chats already have embeddings, or there are no chats yet.")
else:
    texts = [combined_text(user_text, assistant_text) for _, user_text, assistant_text in rows]
    embeddings = await embed_texts(texts)
    if isinstance(embeddings, dict) and 'error' in embeddings:
        print(embeddings['error'])
    else:
        for (row_id, _, _), embedding in zip(rows, embeddings):
            conn.execute(
                "UPDATE conversations SET embedding_json = ? WHERE id = ?",
                (json.dumps(embedding), row_id),
            )
        conn.commit()
        print(f"Added embeddings to {len(rows)} saved chats.")

## Step 13: Retrieve the most relevant past chats by meaning

Now we embed the new question, compare it to the saved memories, and pull back the most similar ones.

In [ ]:
QUERY = "What study method fits someone who likes flash cards and biology?"
TOP_K = 2

rows = conn.execute(
    "SELECT id, label, user_text, assistant_text, embedding_json FROM conversations WHERE embedding_json IS NOT NULL"
).fetchall()

if not rows:
    print("No embedded memories yet. Run Steps 7, 11, and 12 first.")
else:
    query_embeddings = await embed_texts(["[CLS] Represent this sentence for searching relevant passages: " + QUERY + " [SEP]"])
    if isinstance(query_embeddings, dict) and 'error' in query_embeddings:
        print(query_embeddings['error'])
    else:
        query_embedding = query_embeddings[0]
        scored = []
        for row in rows:
            memory_embedding = json.loads(row[4])
            score = cosine_similarity(query_embedding, memory_embedding)
            scored.append((score, row))
        scored.sort(reverse=True, key=lambda item: item[0])
        top_rows = scored[:TOP_K]
        print("Most similar saved chats:")
        for score, row in top_rows:
            print(f"score={score:.3f} | #{row[0]} | {row[1]}")
            print("  user:      " + row[2])
            print("  assistant: " + row[3])
            print()

## Step 14: Use relevant memories in a new answer

The final step turns the retrieved memories back into `messages` and appends the new question.

In [ ]:
if not rows:
    print("No embedded memories yet. Run Steps 7, 11, 12, and 13 first.")
else:
    memory_messages = []
    for score, row in top_rows:
        memory_messages.append({"role": "user", "content": row[2]})
        memory_messages.append({"role": "assistant", "content": row[3]})
    memory_messages.append({"role": "user", "content": QUERY})
    reply = await chat_completion(memory_messages)
    if 'error' in reply:
        print(reply['error'])
    else:
        print(reply['choices'][0]['message']['content'])

## Your turn

Add a few more facts about yourself, save them, and see whether the last-few-chats method or the meaning-based method gives better memory for your questions. Remember that all of this memory lives only in this browser.